# Train the ISL sign classifier on Kaggle

Runs the whole pipeline for one vocabulary:
download INCLUDE videos → MediaPipe landmarks + augmentation → train → evaluate → export for the web app.

**Before running**
1. Settings → **Internet: on**. This needs a phone-verified Kaggle account.
2. Accelerator: **None (CPU)** is fine. Training takes ~20 min on CPU, and a GPU doesn't speed up the slow step (MediaPipe).
3. Choose `CONFIG` below, then **Save Version → Save & Run All (Commit)**. The run continues with the browser closed.
   Run it once per config (the 24-sign and INCLUDE-50 runs as separate versions) so each stays well within Kaggle's session limit.

**Output** (*Output* tab of the saved version): `results/<name>/` holds `best.pt`, the test/val metrics,
confusion matrices, training curves, `web/<name>.onnx` + `.json` for the app, and `landmarks.zip`
(the extracted features, so you can retrain later without re-running MediaPipe).

In [ ]:
# ---- settings ----
CONFIG = "configs/include50.yaml"     # or "configs/isl24.yaml"
REPO = "https://github.com/reubenjrouse/indian-sign-language-translation.git"
BRANCH = "main"

In [ ]:
import os, shutil, subprocess
print("CPUs:", os.cpu_count())
for d in ("/kaggle/working", "/tmp"):
    if os.path.exists(d):
        print(d, "free GB:", round(shutil.disk_usage(d).free / 1e9, 1))

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone --depth 1 -b {BRANCH} {REPO} repo
%cd /kaggle/working/repo
# Kaggle already has torch, numpy, opencv, sklearn, matplotlib; add the rest.
!pip install -q mediapipe==1.1.0 onnx onnxruntime onnxscript httpx python-dotenv pyyaml tqdm
!python -c "import mediapipe, torch; print('mediapipe', mediapipe.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# Videos and landmarks go to /tmp (large, not saved); results go to /kaggle/working (saved as output).
workers = max(1, os.cpu_count())
!python scripts/run_all.py --config {CONFIG} --data-dir /tmp/isl --out /kaggle/working/results --workers {workers}

In [ ]:
import json, glob
from IPython.display import Image, display
print(json.dumps(json.load(open("/kaggle/working/results/summary.json")), indent=2))
for png in sorted(glob.glob("/kaggle/working/results/*/*.png")):
    print(png); display(Image(png, width=700))

In [ ]:
# Keep the output small: drop the cloned repo.
%cd /kaggle/working
!rm -rf repo
!du -sh /kaggle/working/results/*